In [2]:
%pip install plotly

     ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.6/14.8 MB 49.9 MB/s eta 0:00:01
     ------------ --------------------------- 4.5/14.8 MB 57.4 MB/s eta 0:00:01
     ---------------------- ----------------- 8.3/14.8 MB 66.4 MB/s eta 0:00:01
     ---------------------------- ---------- 10.7/14.8 MB 72.6 MB/s eta 0:00:01
     ---------------------------------- ---- 13.0/14.8 MB 73.1 MB/s eta 0:00:01
     --------------------------------------  14.8/14.8 MB 59.5 MB/s eta 0:00:01
     --------------------------------------  14.8/14.8 MB 59.5 MB/s eta 0:00:01
     --------------------------------------- 14.8/14.8 MB 38.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/308.2 kB ? eta -:--:--
     ------------------------------------- 308.2/308.2 kB 19.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install tqdm

     ---------------------------------------- 0.0/78.5 kB ? eta -:--:--
     ---------------------------------------- 78.5/78.5 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install nbformat


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from collections import Counter

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm

In [5]:
# https://drive.google.com/file/d/10ioidzd1OgJrssL2XzC9XCe6R6Ham_2G/view
# agenda_items = pd.read_csv('../data/currentTermAgendaItems.csv')
# https://drive.google.com/file/d/1OfJNVo0dz9Xz0IrQXXe967dtRZG7cQne/view?usp=drive_link
agenda_items = pd.read_csv('../data/allAgendaItems.csv')

## Exploring all fields first...

In [6]:
agenda_items.head(5)

,id,termId,agendaItemId,councilAgendaItemId,decisionBodyId,meetingId,itemProcessId,decisionBodyName,meetingDate,reference,...,decisionRecommendations,decisionAdvice,subjectTerms,wardId,backgroundAttachmentId,agendaItemAddress,address,geoLocation,planningApplicationNumber,neighbourhoodId
0,1da8524e-019a-4acc-8e26-9f9abd2351bb,4,42609,42609,261,6809,12,City Council,1367899200000,2013.BL33.1,...,"<div class=""WordSection1""><div><p class=""MsoNo...",NaN,"by-laws, draft by-laws; bylaws, draft bylaws","{1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...",{58067},[],NaN,NaN,NaN,NaN
1,95c19901-8c03-4b34-92f5-b312028cf51c,4,42078,42360,261,6809,12,City Council,1367899200000,2013.CA21.2,...,"<div class=""WordSection1""><div><div><p style=""...",NaN,;,"{1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...","{57548,57455}",[],NaN,NaN,NaN,NaN
2,2fbd9280-2628-448f-90d5-0dad073bc3c5,4,42079,42361,261,6809,12,City Council,1367899200000,2013.CA21.3,...,"<div class=""WordSection1""><div><div><div><div>...",NaN,;,"{1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...","{57456,57457,57549}",[],NaN,NaN,NaN,NaN
3,fcfd2901-7b0e-4e3d-9866-69bbc7b4dd46,4,42080,42362,261,6809,12,City Council,1367899200000,2013.CA21.4,...,"<div class=""WordSection1""><div><div><div><p st...",NaN,;,"{1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,...","{57550,57458}",[],NaN,NaN,NaN,NaN
4,a408b557-aed7-42ab-9d54-8bac897e1f64,4,32434,32434,369,4520,6,Planning and Growth Management Committee,1320728400000,2011.PG9.8,...,"<div class=""WordSection1""><div><p>The Planning...",NaN,signs; signage,{35},{41926},"[{""agendaItemId"":32434,""addressId"":16902,""stre...","{""2787 Eglinton Toronto Ontario ""}","{""43.7370323,-79.2462529""}",NaN,{39}


In [4]:
agenda_items.columns

Index(['id', 'termId', 'agendaItemId', 'councilAgendaItemId', 'decisionBodyId',
       'meetingId', 'itemProcessId', 'decisionBodyName', 'meetingDate',
       'reference', 'termYear', 'agendaCd', 'meetingNumber', 'itemStatus',
       'agendaItemTitle', 'agendaItemSummary', 'agendaItemRecommendation',
       'decisionRecommendations', 'decisionAdvice', 'subjectTerms', 'wardId',
       'backgroundAttachmentId', 'agendaItemAddress', 'address', 'geoLocation',
       'planningApplicationNumber', 'neighbourhoodId', 'textSearchVector'],
      dtype='object')

In [8]:
np.sum(agenda_items['termId'].apply(lambda x: int(pd.isna(x))))

0

In [9]:
# Count the number of nulls / NaNs
counts = {}
for col in agenda_items.columns:
    # counts[col] = np.sum(agenda_items[col].apply(lambda x: int(pd.isna(x)))).item()
    counts[col] = agenda_items.shape[0] - agenda_items[col].count()
    # next time: could just do agenda_items.shape[0] - agenda_items[col].count()
counts

{'id': 0,
 'termId': 0,
 'agendaItemId': 0,
 'councilAgendaItemId': 0,
 'decisionBodyId': 0,
 'meetingId': 0,
 'itemProcessId': 0,
 'decisionBodyName': 0,
 'meetingDate': 0,
 'reference': 0,
 'termYear': 0,
 'agendaCd': 33,
 'meetingNumber': 0,
 'itemStatus': 0,
 'agendaItemTitle': 0,
 'agendaItemSummary': 8,
 'agendaItemRecommendation': 12186,
 'decisionRecommendations': 3527,
 'decisionAdvice': 75201,
 'subjectTerms': 0,
 'wardId': 5815,
 'backgroundAttachmentId': 4044,
 'agendaItemAddress': 0,
 'address': 44911,
 'geoLocation': 44914,
 'planningApplicationNumber': 79324,
 'neighbourhoodId': 53762}

In [10]:
agenda_items.shape

(87454, 27)

In [14]:
normalized_counts = {field: counts[field] / agenda_items.shape[0] * 100 for field in counts}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_counts],
        y=[normalized_counts[field] for field in normalized_counts],
        # textposition='inside',
        text=[str(round(normalized_counts[field], 1)) for field in normalized_counts],
        # text_auto="True"
        # title='Missingness of Agenda Item Fields',
    )],
    layout=go.Layout(
        title={'text': 'Missingness of Agenda Item Fields (All Items)', 'subtitle': {'text': 'Or, the amount of null values.'}},
        # subtitle_text='Or, the amount of null values',
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}, 'range': [0, 105]}
    )
    # layout_title_text='Missingness of Agenda Item Fields',
    # x_axis={'title': {'text': 'Field'}}
)
fig

In [15]:
unique_counts = {}
for field in agenda_items.columns:
    unique_counts[field] = len(set(agenda_items[field]))
unique_counts


{'id': 87454,
 'termId': 5,
 'agendaItemId': 70502,
 'councilAgendaItemId': 87454,
 'decisionBodyId': 324,
 'meetingId': 5806,
 'itemProcessId': 6,
 'decisionBodyName': 169,
 'meetingDate': 3087,
 'reference': 70502,
 'termYear': 19,
 'agendaCd': 148,
 'meetingNumber': 213,
 'itemStatus': 16,
 'agendaItemTitle': 58613,
 'agendaItemSummary': 63134,
 'agendaItemRecommendation': 54769,
 'decisionRecommendations': 80496,
 'decisionAdvice': 10291,
 'subjectTerms': 22448,
 'wardId': 1506,
 'backgroundAttachmentId': 66704,
 'agendaItemAddress': 33563,
 'address': 22457,
 'geoLocation': 26402,
 'planningApplicationNumber': 3799,
 'neighbourhoodId': 1217}

In [ ]:
np.sum(agenda_items['agendaCd'].apply(lambda x: int(not pd.isna(x))))

np.int64(10402)

In [16]:
normalized_uniqueness = {field: unique_counts[field] / agenda_items.shape[0] * 100 for field in counts}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_uniqueness],
        y=[normalized_uniqueness[field] for field in normalized_uniqueness],
        text=[str(round(normalized_uniqueness[field], 1)) for field in normalized_uniqueness],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Agenda Item Fields (All Items)'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}}
    )
)
fig

In [17]:
unique_counts_excl_na = {}
for field in agenda_items.columns:
    # Gather unique values
    unique_values = set(agenda_items[field])
    unique_values.discard(np.nan)
    
    # Field: (# unique non-null values, total # non-null values)
    unique_counts_excl_na[field] = len(unique_values), agenda_items.shape[0] - counts[field]
unique_counts_excl_na


{'id': (87454, 87454),
 'termId': (5, 87454),
 'agendaItemId': (70502, 87454),
 'councilAgendaItemId': (87454, 87454),
 'decisionBodyId': (324, 87454),
 'meetingId': (5806, 87454),
 'itemProcessId': (6, 87454),
 'decisionBodyName': (169, 87454),
 'meetingDate': (3087, 87454),
 'reference': (70502, 87454),
 'termYear': (19, 87454),
 'agendaCd': (147, 87421),
 'meetingNumber': (213, 87454),
 'itemStatus': (16, 87454),
 'agendaItemTitle': (58613, 87454),
 'agendaItemSummary': (63133, 87446),
 'agendaItemRecommendation': (54768, 75268),
 'decisionRecommendations': (80495, 83927),
 'decisionAdvice': (10290, 12253),
 'subjectTerms': (22448, 87454),
 'wardId': (1505, 81639),
 'backgroundAttachmentId': (66703, 83410),
 'agendaItemAddress': (33563, 87454),
 'address': (22456, 42543),
 'geoLocation': (26401, 42540),
 'planningApplicationNumber': (3798, 8130),
 'neighbourhoodId': (1216, 33692)}

In [18]:
normalized_uniqueness_excl_na = {field: unique_counts_excl_na[field][0] / unique_counts_excl_na[field][1] * 100 for field in unique_counts_excl_na}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_uniqueness_excl_na],
        y=[normalized_uniqueness_excl_na[field] for field in normalized_uniqueness_excl_na],
        text=[str(round(normalized_uniqueness_excl_na[field], 1)) for field in normalized_uniqueness_excl_na],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Agenda Item Fields, excluding null values (All Items)'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}}
    )
)
fig

## Moving onto subject terms themselves...

In [84]:
agenda_items['subjectTerms'].count()

np.int64(10412)

In [19]:
subject_terms_semicolon_sep = []
subject_terms_comma_sep = []
for i in range(agenda_items.shape[0]):
    subject_terms_semicolon_sep += [value.strip() for value in agenda_items['subjectTerms'].iloc[i].split(';') if value.strip()]
for term in subject_terms_semicolon_sep:
    subject_terms_comma_sep += [value.strip() for value in term.split(',') if value.strip()]
print("len(semicolon-separated):", len(subject_terms_semicolon_sep))
print("len(semicolon-separated, unique):", len(set(subject_terms_semicolon_sep)))
print("len(comma-separated):", len(subject_terms_comma_sep))
print("len(comma-separated, unique):", len(set(subject_terms_comma_sep)))

len(semicolon-separated): 380701
len(semicolon-separated, unique): 38168
len(comma-separated): 523917
len(comma-separated, unique): 4963


In [20]:
subject_terms_semicolon_sep[1000:1010]

['temporary sign permits, community events',
 'community festivals',
 'cultural events',
 'events',
 'festivals',
 'by-laws',
 'bylaws',
 'fences',
 'division fences',
 'line fences']

In [21]:
with open("semicolon_separated_subject_terms.txt", 'w', encoding='utf-8') as file:
    for term in subject_terms_semicolon_sep:
        file.write(term)
        file.write("\n")
with open("comma_separated_subject_terms.txt", 'w', encoding='utf-8') as file:
    for term in subject_terms_comma_sep:
        file.write(term)
        file.write("\n")
with open("semicolon_separated_subject_terms_unique.txt", 'w', encoding='utf-8') as file:
    for term in sorted(set(subject_terms_semicolon_sep)):
        file.write(term)
        file.write("\n")
with open("comma_separated_subject_terms_unique.txt", 'w', encoding='utf-8') as file:
    for term in sorted(set(subject_terms_comma_sep)):
        file.write(term)
        file.write("\n")

In [22]:
agenda_items.iloc[10408]

id                                        92924668-24ee-4e63-872d-de5cad97a862
termId                                                                       3
agendaItemId                                                             16335
councilAgendaItemId                                                      16335
decisionBodyId                                                               2
meetingId                                                                 2206
itemProcessId                                                               12
decisionBodyName                                                  City Council
meetingDate                                                      1243224000000
reference                                                          2009.MM36.3
termYear                                                                  2009
agendaCd                                                                    MM
meetingNumber                                       

In [23]:
term_uniqueness = [
    len(set(subject_terms_semicolon_sep)) / len(subject_terms_semicolon_sep),
    len(set(subject_terms_comma_sep)) / len(subject_terms_comma_sep)
]
term_uniqueness = [val * 100 for val in term_uniqueness]

# Graph uniqueness
fig = go.Figure(
    data=[go.Bar(
        x=['Semicolon-separated terms', 'Comma-separated terms'],
        y=term_uniqueness,
        text=[str(round(val, 1)) for val in term_uniqueness],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Subject Terms'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}, 'range': [0, 105]}
    )
)
fig

In [25]:
# Graph uniqueness
plot_df = pd.DataFrame(
    [
        ['Semicolon-separated', len(set(subject_terms_semicolon_sep)), len(subject_terms_semicolon_sep) - len(set(subject_terms_semicolon_sep))],
        ['Comma-separated', len(set(subject_terms_comma_sep)), len(subject_terms_comma_sep) - len(set(subject_terms_comma_sep))]
    ],
    columns=['Subject Terms', 'Unique', 'Duplicate'])
fig = px.bar(
    plot_df,
    x='Subject Terms',
    y=['Unique', 'Duplicate'],
    title='Uniqueness of Subject Terms (All Items)',
    text_auto=True,
    # text=[{'Unique': 1, 'Duplicate': 2}, {'Unique': 3, 'Duplicate': 4}],
    height=600,
    width=550,
)
fig.show()

### Calculate overlap of subject terms with City Subject Thesaurus

In [26]:
# https://open.toronto.ca/dataset/city-subject-thesaurus/
city_subject_thesaurus = pd.read_csv('../data/City Subject Thesaurus (xls).csv')
city_subject_thesaurus

,Name,URI,Identifier,Definition(s),Source,Synonym(s),Status,Broader concept,Narrower concept,Related concept,Legacy ID
0,[actions of government],http://vocab.toronto.ca/id/100783,100783,placeholder,developed by editor,NaN,approved,[activities in government],"business travel (municipal), city initiatives,...",NaN,3926
1,[activities in business and industry],http://vocab.toronto.ca/id/100192,100192,placeholder,"Developed by editors. (Jan 21, 2008)",NaN,approved,business & industry (sc),"business registration, business start-up, cons...",NaN,3196
2,[activities in community and life],http://vocab.toronto.ca/id/101082,101082,placeholder,"Developed by editors. (Mar 12, 2008)",NaN,approved,community & life (sc),"adoption, child care, child custody, child sup...",NaN,3590
3,[activities in culture],http://vocab.toronto.ca/id/100188,100188,placeholder,"Developed by editors. (Mar 12, 2008)",NaN,approved,culture (sc),arts and culture,NaN,3581
4,[activities in education],http://vocab.toronto.ca/id/100933,100933,placeholder,"Developed by editors. (Mar 5, 2008)",NaN,approved,education (sc),"adult and community education, early childhood...","[activities in education], public education",3490
...,...,...,...,...,...,...,...,...,...,...,...
2507,zoning,http://vocab.toronto.ca/id/100603,100603,the government regulation of land and building...,From Access Toronto Kb document (Building  Zo...,NaN,approved,[activities in planning and development],"interim control, part lot control","rezoning, zoning, zoning bylaws, zoning design...",3338
2508,zoning bylaws,http://vocab.toronto.ca/id/100601,100601,"Municipal laws that regulate the use, size, he...",From Access Toronto Kb document (Building  Zo...,"zoning by-laws, zoning bylaw project, zoning p...",approved,[rules in planning and development],NaN,"interim control, land use, minor variances, of...",680
2509,zoning designations,http://vocab.toronto.ca/id/100607,100607,"the controls outlined in the zoning bylaw, for...","City web page, Toronto Building, Customer Serv...","permitted use requests, permitted uses",approved,[attributes in property],NaN,"land use, rezoning, zoning, zoning bylaws, zon...",684
2510,zoo animals,http://vocab.toronto.ca/id/101369,101369,placeholder,from Access Toronto Kb document (Parks - High ...,NaN,approved,[objects in recreation and tourism],NaN,"zoo animals, zoos",1123


In [97]:
len(set(city_subject_thesaurus['Name']))

2512

In [27]:
cst = set(city_subject_thesaurus['Name'])
comma_st = set(subject_terms_comma_sep)
semicolon_st = set(subject_terms_semicolon_sep)
print('cst comma semicolon')
print(len(cst), len(comma_st), len(semicolon_st))

cst comma semicolon
2512 4963 38168


In [28]:
len(comma_st & cst)

1903

In [29]:
len(comma_st & semicolon_st)

3833

In [30]:
len(semicolon_st & cst)

1035

In [31]:
len(cst & semicolon_st & comma_st)

1033

In [32]:
len(cst - semicolon_st - comma_st)

607

In [34]:
1903-1033

870

In [108]:
1312+316+1198-314


2512

In [109]:
1198-314

884

In [110]:
2+314+884+1312

2512

In [111]:
2083-314

1769

In [35]:
len(semicolon_st & comma_st - (semicolon_st & comma_st & cst))

2800

In [37]:
len(comma_st - cst - semicolon_st)

260

In [36]:
len(semicolon_st - cst - comma_st)

34333

In [38]:
len(comma_st - cst)

3060

In [119]:
len(semicolon_st - comma_st)

5997

In [39]:
len(cst - comma_st)

609

In [41]:
len(semicolon_st - comma_st)

34335